In [1]:
import pandas as pd

df = pd.read_csv("./dataset/support_tickets.csv")

df = df[['query_text', 'category', 'sentiment']].dropna()

df.head()

,query_text,category,sentiment
0,I see a transaction of ₹999 I didn't make on 2...,Fraud/Unauthorized,Urgent
1,I see a transaction of ₹500 I didn't make on 1...,Fraud/Unauthorized,Neutral
2,"I see a transaction of ₹25,000 I didn't make o...",Fraud/Unauthorized,Neutral
3,I see a transaction of ₹999 I didn't make on 1...,Fraud/Unauthorized,Confused
4,"I see a transaction of ₹25,000 I didn't make o...",Fraud/Unauthorized,Frustrated


In [2]:
from utils.clean_text import clean_text

df['clean_text'] = df['query_text'].apply(clean_text)

In [3]:
df['category'].unique()

array(['Fraud/Unauthorized', 'Loan', 'KYC', 'Account Access'],
      dtype=object)

In [4]:
from sklearn.preprocessing import LabelEncoder

intent_encoder = LabelEncoder()
df['intent'] = intent_encoder.fit_transform(df['category'])

sentiment_encoder = LabelEncoder()
df['sentiment_label'] = sentiment_encoder.fit_transform(df['sentiment'])

In [5]:
from sklearn.model_selection import train_test_split

x = df['clean_text']
y_intent = df['intent']
y_sentiment = df['sentiment_label']

x_train, x_test, y_intent_train, y_intent_test, y_sent_train, y_sent_test = train_test_split(
    x, y_intent, y_sentiment,
    test_size=0.2,
    random_state=42,
    stratify=y_intent
)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

intent_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

intent_model.fit(x_train_vec, y_intent_train)

y_pred_intent = intent_model.predict(x_test_vec)

print("Intent Accuracy:", accuracy_score(y_intent_test, y_pred_intent))
print(classification_report(y_intent_test, y_pred_intent))

Intent Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [8]:
sentiment_model = LogisticRegression(max_iter=1000)

sentiment_model.fit(x_train_vec, y_sent_train)

y_pred_sent = sentiment_model.predict(x_test_vec)

print("Sentiment Accuracy:", accuracy_score(y_sent_test, y_pred_sent))

Sentiment Accuracy: 0.175


In [9]:
import numpy as np

def predict(query):
    query = clean_text(query)
    vec = vectorizer.transform([query])
    
    intent_probs = intent_model.predict_proba(vec)[0]
    sentiment_probs = sentiment_model.predict_proba(vec)[0]
    
    intent_idx = np.argmax(intent_probs)
    sentiment_idx = np.argmax(sentiment_probs)
    
    intent_conf = intent_probs[intent_idx]
    sentiment_conf = sentiment_probs[sentiment_idx]
    
    intent = intent_encoder.inverse_transform([intent_idx])[0]
    sentiment = sentiment_encoder.inverse_transform([sentiment_idx])[0]
    
    return {
        "intent": intent,
        "intent_confidence": float(intent_conf),
        "sentiment": sentiment,
        "sentiment_confidence": float(sentiment_conf)
    }

predict("I see an unknown transaction on my account")

{'intent': 'Fraud/Unauthorized',
 'intent_confidence': 0.4051035850894551,
 'sentiment': 'Frustrated',
 'sentiment_confidence': 0.19229809749536153}

In [10]:
import pickle
import os

os.makedirs("./models", exist_ok=True)

pickle.dump(intent_model, open("./models/intent_model.pkl", "wb"))
pickle.dump(sentiment_model, open("./models/sentiment_model.pkl", "wb"))
pickle.dump(vectorizer, open("./models/vectorizer.pkl", "wb"))
pickle.dump(intent_encoder, open("./models/intent_encoder.pkl", "wb"))
pickle.dump(sentiment_encoder, open("./models/sentiment_encoder.pkl", "wb"))